# DECKBench: User Simulator Tutorial

This notebook walks through how the **DECKBench user simulator** works — from design philosophy to running your own simulation. It is intended to complement Section 4.3 of the paper and give researchers everything they need to:

1. Understand *why* the simulator is designed the way it is
2. Understand *how* the three personas are defined and behave differently
3. Run the simulator on their own paper–slide pairs
4. Inspect and validate the generated editing instructions
5. Swap in a custom simulator or editor agent

---

**Prerequisites**: Complete the installation steps in the [README](https://github.com/morgan-heisler/DeckBench/blob/main/README.md), including cloning the repo, installing dependencies, and setting your `OPENAI_API_KEY` (or equivalent).

```
git clone https://github.com/morgan-heisler/DeckBench.git
cd DeckBench
pip install -r requirements.txt
```

## 1. Background: Why a Simulated User?

Evaluating multi-turn slide editing requires a *sequence* of realistic editing instructions — not a single query. Collecting these from real users at scale is:
- **Expensive** (requires domain experts who can read research papers)
- **Irreproducible** (different users produce different instruction sequences)
- **Slow** (human annotation cannot run in parallel with model experiments)

DECKBench addresses this with a **simulated user agent**: an LLM that observes the *current* slide deck and the *ground-truth* slide deck, then generates a natural-language editing instruction that describes what a real user might ask to move the current deck closer to the ground truth.

```
┌─────────────────────────────────────────────────────────┐
│  Ground-Truth Deck  ──┐                                 │
│                       ▼                                 │
│             [ User Simulator LLM ]                      │
│                       │                                 │
│  Current Deck ────────┘                                 │
│                       │                                 │
│                       ▼                                 │
│         Editing Instruction (natural language)          │
│                       │                                 │
│                       ▼                                 │
│             [ Editor Agent LLM ]                        │
│                       │                                 │
│                       ▼                                 │
│                 Updated Deck  ──► next turn             │
└─────────────────────────────────────────────────────────┘
```

This design is inspired by the τ²-Bench framework (Barres et al., 2025) and adapted for the structured, multi-modal nature of slide editing.

## 2. The Three Personas

A single canonical instruction format would bias evaluation toward models that happen to match that style. Instead, DECKBench defines **three personas** that differ in verbosity, specificity, and tone. This tests robustness across realistic user communication styles.

| Persona | Verbosity | Instruction Style | Typical Length |
|---|---|---|---|
| **Granular Analyst** | High | Explicit — lists exact slide titles, bullet text, figure captions, example numbers | 100–400 words |
| **Balanced Editor** | Medium | Actionable — slide/bullet-level guidance, some phrasing suggestions, max 4 sentences | 30–80 words |
| **Executive** | Low | Strategic — one sentence describing the conceptual or structural change only | 10–25 words |

### Persona Prompt Configurations

These are the exact configurations used in the paper (Appendix A):

In [ ]:
# The three persona configurations used in DECKBench
# These are passed to the simulator's system prompt to control behaviour.

PERSONAS = {
    "granular_analyst": {
        "name": "Granular Analyst",
        "prompt_verbosity": "detailed",
        "prompt_detail_level": (
            "generate prompt with explicit instructions including the slide titles, "
            "bullet points, table rows, captions, and example numbers necessary "
            "for replicating the slide accurately"
        ),
        "prompt_tone": "methodical, precise",
        "restrictions": [
            "Do not omit any necessary detail for replicating the slide accurately.",
            "Always include concrete guidance for content, structure, and phrasing.",
            "Do not leave prompts ambiguous or open-ended.",
            "English only.",
        ],
    },
    "balanced_editor": {
        "name": "Balanced Editor",
        "prompt_verbosity": "medium",
        "prompt_detail_level": (
            "clear, actionable prompt at slide or bullet level; may include some "
            "suggestions for phrasing but no full tables or exact numbers in the "
            "prompt itself; maximum 4 sentences in the prompt"
        ),
        "prompt_tone": "efficient, professional",
        "restrictions": [
            "Do not reference files the downstream assistant cannot access.",
            "Avoid lengthy paragraphs; keep prompt focused on clarity and actionable guidance.",
            "English only.",
        ],
    },
    "executive": {
        "name": "Executive",
        "prompt_verbosity": "short and concise",
        "prompt_detail_level": "generate a very succinct prompt; only one sentence of prompt.",
        "prompt_tone": "brief, strategic, professional",
        "restrictions": [
            "Do not reference any external files or resources the downstream assistant cannot access.",
            "Do not omit any necessary information for replicating the slide accurately.",
            "Do not give step-by-step instructions; only describe the conceptual or structural change.",
            "English only.",
        ],
    },
}

for key, persona in PERSONAS.items():
    print(f"Persona key : {key}")
    print(f"  Name      : {persona['name']}")
    print(f"  Verbosity : {persona['prompt_verbosity']}")
    print(f"  Tone      : {persona['prompt_tone']}")
    print()

## 3. Environment Setup

In [ ]:
import os
import json
import subprocess
from pathlib import Path

# ── Set your API key ──────────────────────────────────────────────────────────
# Option A: already set in your shell environment
#   export OPENAI_API_KEY="sk-..."
# Option B: set here (not recommended for shared notebooks)
#   os.environ["OPENAI_API_KEY"] = "sk-..."

api_key = os.environ.get("OPENAI_API_KEY", None)
if api_key is None:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. "
        "Run: export OPENAI_API_KEY='sk-...' before launching this notebook."
    )
print(f"API key found: {api_key[:8]}...")

# ── Verify we are inside the cloned repo ──────────────────────────────────────
REPO_ROOT = Path(".").resolve()
assert (REPO_ROOT / "simulation_pipeline").exists(), (
    f"Could not find simulation_pipeline/ in {REPO_ROOT}. "
    "Please run this notebook from the root of the DeckBench repository."
)
print(f"Repository root : {REPO_ROOT}")

## 4. Quick-Start: Running the Simulator on a Single Deck

The fastest way to see the simulator in action is to run it on one paper–slide pair with the default `balanced_editor` persona for 2 turns.

In [ ]:
# ── Paths — update these to point to your data ───────────────────────────────
GT_SLIDES_ROOT  = "/path/to/ref_slides"      # ground-truth slide deck PDFs
INITIAL_DECKS   = "/path/to/gen_slides"      # initial generated slide deck PDFs
SIMULATION_NAME = "tutorial_run_1"
MAX_TURNS       = 2
PERSONA         = "balanced_editor"          # try: granular_analyst, executive
CONFIG_PATH     = "simulation_pipeline/custom/config.yaml"

cmd = [
    "python", "simulation_pipeline/multiturn_simulation.py",
    "--data_path.gt_slides_root",  GT_SLIDES_ROOT,
    "--data_path.deck_list_path",  INITIAL_DECKS,
    "--simulation.simulation_name", SIMULATION_NAME,
    "--simulation.max_turns",      str(MAX_TURNS),
    "--user_agent.persona_name",   PERSONA,
    "--config",                    CONFIG_PATH,
]

print("Running command:")
print(" ".join(cmd))
print()

# Uncomment to execute:
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr)

## 5. What the Simulator Actually Does — Step by Step

This section demystifies the simulator's internals. Each turn proceeds as follows:

In [ ]:
# This cell shows a minimal, self-contained re-implementation of the
# simulator's core logic so you can trace exactly what happens.
# It does NOT require the full simulation pipeline to be installed.

from openai import OpenAI
client = OpenAI()  # reads OPENAI_API_KEY from environment


def build_simulator_system_prompt(persona: dict) -> str:
    """Build the system prompt for the user simulator from a persona config."""
    restrictions_text = "\n".join(f"- {r}" for r in persona["restrictions"])
    return (
        f"You are simulating a user who is editing an academic slide deck.\n"
        f"Your task is to generate a single editing instruction.\n\n"
        f"Communication style: {persona['prompt_verbosity']}\n"
        f"Detail level: {persona['prompt_detail_level']}\n"
        f"Tone: {persona['prompt_tone']}\n\n"
        f"Rules:\n{restrictions_text}\n\n"
        f"You will be shown:\n"
        f"  1. The CURRENT slide deck content (what the editor has now).\n"
        f"  2. The TARGET slide deck content (what the final deck should look like).\n\n"
        f"Generate ONE editing instruction that describes the most important change "
        f"needed to move the current deck toward the target deck. "
        f"Do not describe all differences — focus on the single most impactful edit."
    )


def build_simulator_user_message(current_text: str, target_text: str) -> str:
    """Build the user message from extracted text of both decks."""
    return (
        f"## CURRENT SLIDE DECK\n{current_text}\n\n"
        f"## TARGET SLIDE DECK\n{target_text}\n\n"
        f"Generate the editing instruction now."
    )


def simulate_one_turn(
    current_deck_text: str,
    target_deck_text: str,
    persona_key: str = "balanced_editor",
    model: str = "gpt-4o",
) -> str:
    """Generate one simulated editing instruction.

    Args:
        current_deck_text : Extracted text from the current (in-progress) deck.
        target_deck_text  : Extracted text from the ground-truth final deck.
        persona_key       : One of 'granular_analyst', 'balanced_editor', 'executive'.
        model             : OpenAI model to use for simulation.

    Returns:
        A natural-language editing instruction string.
    """
    persona = PERSONAS[persona_key]
    system_prompt = build_simulator_system_prompt(persona)
    user_message = build_simulator_user_message(current_deck_text, target_deck_text)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.7,
    )
    return response.choices[0].message.content.strip()


print("Functions defined. See next cell for a live example.")

## 6. Live Example: Same Diff, Three Personas

The following cell demonstrates how the **same pair of decks** produces very different instructions depending on persona — which is exactly the point.

In [ ]:
# Toy example: replace with real extracted text from your decks.
# In practice, text is extracted from PDFs using the pipeline's PDF parser.

CURRENT_DECK_TEXT = """
Slide 1: Introduction
- Deep learning has achieved impressive results
- We propose a new method

Slide 2: Method
- Our architecture uses transformers
- Trained on ImageNet

Slide 3: Results
- We outperform baselines
"""

TARGET_DECK_TEXT = """
Slide 1: Introduction
- Deep learning has achieved impressive results on vision tasks
- Existing methods struggle with long-range dependencies
- We propose ViT-XL: a scalable vision transformer

Slide 2: Method
- Our architecture uses a hierarchical transformer with local attention
- Trained on ImageNet-21k with strong augmentation
- Key innovation: sliding window attention reduces O(n^2) cost

Slide 3: Results
- Top-1 accuracy: 89.4% on ImageNet
- +2.1% over previous SOTA
- 3x faster inference than ViT-L
"""

for persona_key in ["executive", "balanced_editor", "granular_analyst"]:
    print(f"{'='*60}")
    print(f"PERSONA: {PERSONAS[persona_key]['name']}")
    print(f"{'='*60}")
    instruction = simulate_one_turn(
        CURRENT_DECK_TEXT,
        TARGET_DECK_TEXT,
        persona_key=persona_key,
    )
    print(instruction)
    print()

## 7. Running a Full Multi-Turn Simulation

The `multiturn_simulation.py` script wraps the above logic into a pipeline that:
1. Iterates over all decks in `deck_list_path`
2. For each deck, generates `max_turns` instructions (one per turn)
3. Passes each instruction to the editor agent
4. Saves the edited deck and the instruction log per turn

### Running all three personas in sequence

In [ ]:
# Update these paths before running
GT_SLIDES_ROOT = "/path/to/ref_slides"
INITIAL_DECKS  = "/path/to/gen_slides"
MAX_TURNS      = 5
CONFIG_PATH    = "simulation_pipeline/custom/config.yaml"

for persona in ["granular_analyst", "balanced_editor", "executive"]:
    sim_name = f"tutorial_{persona}"
    cmd = [
        "python", "simulation_pipeline/multiturn_simulation.py",
        "--data_path.gt_slides_root",   GT_SLIDES_ROOT,
        "--data_path.deck_list_path",   INITIAL_DECKS,
        "--simulation.simulation_name", sim_name,
        "--simulation.max_turns",       str(MAX_TURNS),
        "--user_agent.persona_name",    persona,
        "--config",                     CONFIG_PATH,
    ]
    print(f"[{persona}] Command:")
    print(" ".join(cmd))
    print()
    # Uncomment to run:
    # subprocess.run(cmd, check=True)

## 8. Inspecting Simulation Output

After running, each simulation produces a folder of per-deck JSON logs. This cell shows how to load and explore them.

In [ ]:
import json
from pathlib import Path

# Update to point at an actual simulation output folder
SIMULATION_OUTPUT_DIR = Path("/path/to/gen_slides/tutorial_balanced_editor")

# Load all instruction logs
instruction_logs = []
for log_file in sorted(SIMULATION_OUTPUT_DIR.glob("**/instructions.json")):
    with open(log_file) as f:
        log = json.load(f)
    instruction_logs.append(log)

print(f"Found {len(instruction_logs)} deck simulation logs")

if instruction_logs:
    example = instruction_logs[0]
    print(f"\nExample log keys: {list(example.keys())}")
    for turn_data in example.get("turns", []):
        turn_num   = turn_data.get("turn", "?")
        instruction = turn_data.get("instruction", "")
        print(f"\n--- Turn {turn_num} ---")
        print(instruction[:300] + ("..." if len(instruction) > 300 else ""))

## 9. Validating Instruction Quality

The paper reports that human annotators rated GPT-5.1-generated instructions as **realistic and actionable in 100% of cases** (250 instructions across 50 decks × 5 turns). The cell below replicates this validation procedure using an LLM judge — useful if you want to validate quality for a different model or persona.

In [ ]:
VALIDATION_SYSTEM_PROMPT = """
You are an expert evaluator of natural-language slide editing instructions.
You will be given a single editing instruction intended for modifying an academic slide deck.

Rate the instruction on two dimensions, each on a scale of 1–5:

1. REALISM (1=clearly artificial/nonsensical, 5=indistinguishable from a real user request)
2. ACTIONABILITY (1=too vague to act on, 5=an editor could carry it out without asking follow-up questions)

Respond ONLY with valid JSON in this exact format:
{"realism": <int>, "actionability": <int>, "comment": "<one sentence>"}
"""


def validate_instruction(instruction: str, judge_model: str = "gpt-4o") -> dict:
    """Rate a simulated editing instruction for realism and actionability."""
    response = client.chat.completions.create(
        model=judge_model,
        messages=[
            {"role": "system", "content": VALIDATION_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Instruction to evaluate:\n\n{instruction}"},
        ],
        temperature=0.0,
    )
    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw": raw, "parse_error": True}


# ── Quick smoke-test with three example instructions ──────────────────────────
example_instructions = [
    # Executive
    "Add a dedicated 'Future Work' slide listing the three envisioned research directions.",
    # Balanced
    "Replace the current 'Problem Statement' slide with a 'Motivation' slide that "
    "opens with the advantages of implicit neural representations before introducing "
    "their limitations on low-end devices.",
    # Granular
    "Add a new slide titled 'Limitations' after the Conclusion slide with exactly "
    "three bullet points: (1) 'Gaussian initialization still needs 1-2 auxiliary MLP "
    "layers'; (2) 'RVQ codebooks add memory overhead at low bitrates'; "
    "(3) 'Trained on single images; no extension yet to video or 3D'.",
]

print(f"{'Instruction (truncated)':<55} {'Realism':>8} {'Action.':>8}  Comment")
print("-" * 100)
for instr in example_instructions:
    result = validate_instruction(instr)
    short  = instr[:52] + "..."
    print(f"{short:<55} {result.get('realism', '?'):>8} "
          f"{result.get('actionability', '?'):>8}  {result.get('comment', '')}")

## 10. Persona Diversity Analysis

The paper reports statistically significant differences across personas in word count, structural editing depth, and semantic focus (ANOVA, all p < 0.001). The cell below reproduces this analysis on your own simulation outputs.

In [ ]:
import re
from scipy import stats


def load_instructions_for_persona(simulation_output_dir: Path) -> list[str]:
    """Load all instruction strings from a simulation output directory."""
    instructions = []
    for log_file in sorted(simulation_output_dir.glob("**/instructions.json")):
        with open(log_file) as f:
            log = json.load(f)
        for turn_data in log.get("turns", []):
            instr = turn_data.get("instruction", "")
            if instr:
                instructions.append(instr)
    return instructions


def word_count(text: str) -> int:
    return len(text.split())


def structural_edit_score(text: str) -> float:
    """Proxy: count explicit structural keywords (slide, bullet, title, section)."""
    keywords = ["slide", "bullet", "title", "section", "heading", "replace", "add", "remove"]
    text_lower = text.lower()
    return sum(text_lower.count(k) for k in keywords)


# ── Update these paths to your simulation outputs ─────────────────────────────
persona_dirs = {
    "granular_analyst": Path("/path/to/gen_slides/tutorial_granular_analyst"),
    "balanced_editor":  Path("/path/to/gen_slides/tutorial_balanced_editor"),
    "executive":        Path("/path/to/gen_slides/tutorial_executive"),
}

word_counts   = {}
struct_scores = {}

for persona, folder in persona_dirs.items():
    if not folder.exists():
        print(f"  [skip] {persona}: folder not found at {folder}")
        continue
    instrs = load_instructions_for_persona(folder)
    word_counts[persona]   = [word_count(i) for i in instrs]
    struct_scores[persona] = [structural_edit_score(i) for i in instrs]
    print(f"{persona}: n={len(instrs)}, "
          f"avg words={sum(word_counts[persona])/len(instrs):.1f}, "
          f"avg struct={sum(struct_scores[persona])/len(instrs):.2f}")

# ANOVA on word count across personas
if len(word_counts) == 3:
    groups = list(word_counts.values())
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"\nWord count ANOVA: F={f_stat:.2f}, p={p_val:.4f}")
    if p_val < 0.001:
        print("✓ Personas produce significantly different word counts (p < 0.001)")

## 11. Cross-Simulator Analysis: Checking for Homogeneous Bias

A potential concern is that using the same model family for both the simulator and the editor (e.g., GPT-5.1 → GPT-5-mini) could introduce **homogeneous bias** — the simulator generating instructions that happen to suit one model's generation style.

The paper addresses this by evaluating both:
- **Homogeneous**: GPT-5.1 (simulator) → GPT-5-mini (editor)
- **Heterogeneous**: Kimi K2 (simulator) → DeepSeek (editor)

The cell below helps you set up and compare these configurations.

In [ ]:
# Configuration table for the paper's cross-simulator experiments
CROSS_SIMULATOR_CONFIGS = [
    {
        "label":        "Homogeneous (GPT family)",
        "user_sim":     "gpt-4.1",        # user simulation model
        "editor":       "gpt-4.1-mini",   # editor model
        "same_family":  True,
    },
    {
        "label":        "Heterogeneous (Kimi → DeepSeek)",
        "user_sim":     "kimi-k2",
        "editor":       "deepseek-v3",
        "same_family":  False,
    },
    {
        "label":        "Heterogeneous (Kimi → Qwen FT)",
        "user_sim":     "kimi-k2",
        "editor":       "qwen-ft",
        "same_family":  False,
    },
]

# Expected finding from the paper:
# ΔDTW and ΔTransSim improve systematically with simulator granularity
# in BOTH homogeneous and heterogeneous settings, ruling out family bias.
PAPER_RESULTS = {
    # (config_label, persona, metric): value after 5 turns
    ("Homogeneous (GPT family)",         "executive",        "ΔDTW"):     0.01006,
    ("Homogeneous (GPT family)",         "balanced_editor",  "ΔDTW"):     0.01682,
    ("Homogeneous (GPT family)",         "granular_analyst", "ΔDTW"):     0.02278,
    ("Heterogeneous (Kimi → DeepSeek)",  "executive",        "ΔDTW"):     0.01101,
    ("Heterogeneous (Kimi → DeepSeek)",  "balanced_editor",  "ΔDTW"):     0.00249,
    ("Heterogeneous (Kimi → DeepSeek)",  "granular_analyst", "ΔDTW"):     0.01549,
}

print("Paper results (ΔDTW after 5 turns):")
print(f"{'Config':<42} {'Persona':<20} {'ΔDTW':>8}")
print("-" * 75)
for (config, persona, metric), val in PAPER_RESULTS.items():
    if metric == "ΔDTW":
        print(f"{config:<42} {persona:<20} {val:>8.5f}")

print()
print("Key finding: ΔDTW increases with granularity in both configurations,")
print("indicating the benchmark is robust across simulator/editor model families.")

## 12. Using a Custom Simulator

You can replace the default simulator with any LLM by implementing the same interface. The key contract is:

```
Input:  (current_deck_text: str, target_deck_text: str, persona: dict) -> instruction: str
```

The cell below shows an example using Anthropic's Claude API:

In [ ]:
# Example: Using Claude as the user simulator
# Requires: pip install anthropic
# Set ANTHROPIC_API_KEY in your environment.

# import anthropic
# claude_client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY

def simulate_one_turn_claude(
    current_deck_text: str,
    target_deck_text: str,
    persona_key: str = "balanced_editor",
    model: str = "claude-sonnet-4-6",
) -> str:
    """Same interface as simulate_one_turn() but using Claude."""
    # import anthropic  # uncomment when ready to use
    persona        = PERSONAS[persona_key]
    system_prompt  = build_simulator_system_prompt(persona)
    user_message   = build_simulator_user_message(current_deck_text, target_deck_text)

    # message = claude_client.messages.create(
    #     model=model,
    #     max_tokens=1024,
    #     system=system_prompt,
    #     messages=[{"role": "user", "content": user_message}],
    # )
    # return message.content[0].text.strip()

    return "[Uncomment the code above and set ANTHROPIC_API_KEY to use Claude.]"


# To integrate with the full simulation pipeline, set the model in
# simulation_pipeline/custom/config.yaml:
#
#   user_agent:
#     model: claude-sonnet-4-6
#     api_service: anthropic
#
# and add your Anthropic API key to the api_keys section of the same config.

print(simulate_one_turn_claude(CURRENT_DECK_TEXT, TARGET_DECK_TEXT))

## 13. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `EnvironmentError: OPENAI_API_KEY not set` | API key not exported | `export OPENAI_API_KEY=sk-...` |
| `AssertionError: Could not find simulation_pipeline/` | Notebook not run from repo root | `cd /path/to/DeckBench` then relaunch |
| `FileNotFoundError` on PDF paths | Path arguments wrong | Check `--data_path.*` point to existing folders |
| Empty `instruction_logs` | Simulation did not complete | Check `result.stderr` from subprocess run |
| Low actionability scores from validator | Persona too terse for the deck diff | Try `granular_analyst` persona for complex diffs |
| HTML → PDF conversion fails | Reveal.js / decktape not installed | Follow [README installation steps](https://github.com/morgan-heisler/DeckBench#installation) |

For persistent issues, please [open a GitHub issue](https://github.com/morgan-heisler/DeckBench/issues).

## 14. Summary

This notebook covered:

| Section | What you learned |
|---|---|
| §1 | Why user simulation is necessary and how it fits the pipeline |
| §2 | The three persona configurations and their behavioural differences |
| §3–4 | Environment setup and quick-start CLI command |
| §5–6 | Step-by-step internals of the simulator and live persona comparison |
| §7 | Running full 5-turn simulations across all personas |
| §8 | Inspecting simulation output logs |
| §9 | Validating instruction quality with an LLM judge |
| §10 | Reproducing the persona diversity ANOVA analysis |
| §11 | Cross-simulator (homogeneous vs. heterogeneous) comparison |
| §12 | Swapping in a custom simulator model |

---

### Citation

If you use DECKBench or this simulator in your research, please cite:

```bibtex
@misc{jang2026deckbench,
  title  = {DECKBench: Benchmarking Multi-Agent Frameworks for Academic Slide Generation and Editing},
  author = {Daesik Jang and Morgan Lindsay Heisler and Linzi Xing and Yifei Li
            and Edward Wang and Ying Xiong and Yong Zhang and Zhenan Fan},
  year   = {2026},
  eprint = {2602.13318},
  archivePrefix = {arXiv},
  primaryClass  = {cs.AI},
  url    = {https://arxiv.org/abs/2602.13318},
}
```